# Week 3 — Spam Classifier\nText preprocessing → TF-IDF → Naive Bayes\n\nBuild a simple SMS/email spam classifier.

In [ ]:
# WEEK 3 — SPAM CLASSIFIER
# Text Preprocessing + TF-IDF + Naive Bayes

import pandas as pd
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. LOAD DATASET
data = [('ham', 'Hey, are we meeting at 5 today?'), ('ham', 'Can you send me the notes from class?'), ('ham', "Don't forget to bring the project file tomorrow."), ('ham', "I'll call you when I reach home."), ('ham', 'Happy birthday! Have a great day.'), ('ham', 'Are you free for lunch today?'), ('ham', 'Please submit the assignment before 6 PM.'), ('ham', 'The meeting has been moved to 3 PM.'), ('ham', 'Can you help me with the Python code?'), ('ham', 'I am on my way to college.'), ('ham', 'Your order has been delivered successfully.'), ('ham', "Let's meet near the library after class."), ('ham', 'Did you finish the Week 2 project?'), ('ham', 'Please remind me about the exam tomorrow.'), ('ham', 'Thanks for helping me yesterday.'), ('ham', 'I will send the documents tonight.'), ('ham', 'What time does the workshop start?'), ('ham', 'See you at the canteen.'), ('ham', 'The teacher shared the assignment details.'), ('ham', 'Call me when you are available.'), ('ham', 'Can we reschedule our meeting to tomorrow?'), ('ham', 'I reached the bus stop.'), ('ham', 'Your package is ready for pickup.'), ('ham', 'Please check the GitHub repository.'), ('ham', 'Good luck for your presentation!'), ('spam', 'Congratulations! You have won a free iPhone. Click now to claim your prize.'), ('spam', 'URGENT! You have won 10000 dollars. Send your bank details to receive it.'), ('spam', 'Win a FREE vacation today! Click the link to claim your reward.'), ('spam', 'You are selected for a cash prize of $5000. Call now.'), ('spam', 'FREE entry! Text WIN to 99999 and get a chance to win big.'), ('spam', 'Claim your exclusive reward before it expires. Click now.'), ('spam', 'Congratulations winner! You have been chosen for a free gift card.'), ('spam', 'Get rich quickly! Invest now and earn guaranteed money.'), ('spam', 'Limited offer! Buy now and get 90 percent discount.'), ('spam', 'You won a lottery prize. Send your details to claim it.'), ('spam', 'FREE recharge available! Click this link immediately.'), ('spam', 'Urgent offer: claim your cash bonus today.'), ('spam', 'You have been selected for a lucky draw. Reply YES now.'), ('spam', 'Win a brand new car! Enter the contest today.'), ('spam', 'Exclusive deal! Get a free coupon by clicking this link.'), ('spam', 'Congratulations! Your number won a cash reward.'), ('spam', 'Act now to receive your free shopping voucher.'), ('spam', 'Special promotion! Earn money from home with no investment.'), ('spam', 'You are a lucky winner. Claim your prize now.'), ('spam', 'Free tickets available for selected customers. Click to claim.'), ('spam', 'Last chance to win a huge cash prize. Reply now.'), ('spam', 'Get 70 percent off today only. Click the offer link.'), ('spam', 'You have won a bonus reward. Verify your account immediately.'), ('spam', 'Congratulations! Claim your free gift before midnight.')]
df = pd.DataFrame(data, columns=["label", "message"])
print("Dataset shape:", df.shape)
display(df.head())

# 2. TEXT PREPROCESSING
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\\S+|www\\S+", " ", text)
    text = re.sub(r"\\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\\s+", " ", text).strip()
    return text

df["clean_message"] = df["message"].apply(clean_text)
display(df[["message", "clean_message"]].head())

# 3. TRAIN / TEST SPLIT
X = df["clean_message"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# 4. TF-IDF
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print("TF-IDF train shape:", X_train_tfidf.shape)

# 5. NAIVE BAYES MODEL
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. EVALUATION
y_pred = model.predict(X_test_tfidf)
print("\nAccuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

# 7. TEST NEW MESSAGES
def predict_spam(message):
    cleaned = clean_text(message)
    vector = vectorizer.transform([cleaned])
    prediction = model.predict(vector)[0]
    confidence = model.predict_proba(vector).max()
    return prediction, confidence

test_messages = [
    "Congratulations! You won a free prize. Click now!",
    "Can you send me the class notes tonight?",
    "URGENT claim your cash reward now!",
    "Let's meet after college at 5 PM."
]

for msg in test_messages:
    prediction, confidence = predict_spam(msg)
    print(f"\\nMessage: {msg}")
    print(f"Prediction: {prediction.upper()}")
    print(f"Confidence: {confidence:.2%}")


## Result\nThe model classifies messages as **spam** or **ham (not spam)** using TF-IDF features and Multinomial Naive Bayes.